# Обработка данных волноводов LiNbO₃

COMSOL → контроль мод → полином n_eff(ω) выбранной степени → JSI и K.

**Выбранный способ:** аппроксимация Re(n_eff) полиномом выбранной степени по угловой частоте ω (рад/фс). По умолчанию используем степень 6; параметр POLYNOMIAL_DEGREE в разделе 4 позволяет выбрать одну степень от 1 до 6. Сравнение степеней сохранено в отдельном ноутбуке «LiNbO3 — расчёт волноводов — сравнения полиномов.ipynb». Этот выбор не заменяет проверку сходимости K.

Для получения исходных данных из терминала в папке проекта:
```powershell
python -m linbo3 --stage comsol --config run_config_comsol.json
```
Ноутбук читает готовые CSV/NPZ и не запускает COMSOL. Графики мод Шмидта и весов отключены.

## 1. Выбор расчёта

По умолчанию выбирается папка COMSOL-прогона с настройками из `run_config_comsol.json` и текущей версией модели. Для работы с другой сохранённой выгрузкой укажи `RESULT_DIR` вручную.

Хранение: подписанные COMSOL-наборы в `results/comsol/`; результаты этого ноутбука находятся в его профиле `analysis`. Новые настройки заменяют прежние данные метода.


In [ ]:
from pathlib import Path
from hashlib import file_digest
import json
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd()
PROJECT_DIR = next((folder for folder in (cwd, *cwd.parents)
                    if (folder / "run_config_comsol.json").is_file()), cwd)
# Если ядро запущено из другой папки, укажи PROJECT_DIR = Path(r"полный путь к проекту")
CONFIG_PATH = PROJECT_DIR / "run_config_comsol.json"
RESULT_DIR = None  # Например Path("results/comsol/<идентификатор-прогона>")

if not CONFIG_PATH.is_file():
    raise FileNotFoundError("Укажи папку LiNbO3 в PROJECT_DIR, в ней должен лежать run_config_comsol.json")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from linbo3.storage import set_profile
set_profile("unclassified")


## 2. Загрузка готовых данных

`pair_status` показывает состояние пар «ширина × диапазон». В `raw_neff` попадают только полностью сохранённые пары. Мнимая часть и пропуски остаются в таблице. Переменная `result_dir` указывает на папку с CSV, NPZ и журналами COMSOL.

In [ ]:
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8-sig"))
output_root = PROJECT_DIR / config["output_dir"]

if RESULT_DIR is not None:
    result_dir = PROJECT_DIR / Path(RESULT_DIR)
    if not (result_dir / "status.json").is_file():
        raise FileNotFoundError(result_dir / "status.json")
else:
    model_path = (PROJECT_DIR / config["model"]).resolve()
    with model_path.open("rb") as model_file:
        model_hash = file_digest(model_file, "sha256").hexdigest()
    candidates = []
    for manifest_file in (output_root / "comsol").glob("*/config.json"):
        if not manifest_file.is_file():
            continue
        manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
        same_config = all(manifest.get(key) == value for key, value in config.items()
                          if key not in {"model", "output_dir", "bands"})
        if (manifest.get("stage") == "comsol" and same_config
                and manifest.get("model_sha256") == model_hash
                and (manifest_file.parent / "status.json").is_file()):
            candidates.append(manifest_file)
    selected = max(candidates, key=lambda file: file.stat().st_mtime_ns, default=None)
    result_dir = selected.parent if selected else None

rows, frames = [], []
if result_dir is not None:
    status = json.loads((result_dir / "status.json").read_text(encoding="utf-8"))
    print(result_dir)
    pair_states = status.get("pairs")
    if pair_states is None:
        manifest = json.loads((result_dir / "config.json").read_text(encoding="utf-8"))
        pair_states = {f"{width:g}": {band: {"state": "complete"} for band in ("PUMP", "IS")}
                       for width in manifest["widths_um"]}
    for width, bands in pair_states.items():
        for band, pair in bands.items():
            state = pair["state"]
            csv_path = result_dir / f"w_{width}" / f"{band}_raw.csv"
            loaded = state == "complete" and csv_path.is_file()
            rows.append({"width_um": float(width), "band": band, "state": state,
                         "loaded": loaded, "error": pair.get("error")})
            if loaded:
                frames.append(pd.read_csv(csv_path).assign(width_um=float(width), band=band))

pair_status = pd.DataFrame(rows, columns=["width_um", "band", "state", "loaded", "error"])
raw_columns = ["width_um", "band", "wavelength_um", "outer_solnum", "solnum", "neff_real", "neff_imag"]
raw_neff = (pd.concat(frames, ignore_index=True).loc[:, raw_columns]
            if frames else pd.DataFrame(columns=raw_columns))

if result_dir is None:
    print("Подходящий прогон ещё не найден. Запусти COMSOL из терминала и повтори эту ячейку.")
else:
    print(f"Загружено пар {len(frames)} из {len(pair_status)}")
    display(pair_status)
    if not raw_neff.empty:
        display(raw_neff.head())

## 3. Re(n_eff), Im(n_eff) и контроль отбора — все ширины

Для каждой ширины показаны оба диапазона PUMP/IS. Серые точки — все решения COMSOL;
зелёные проходят `abs(Im(n_eff)) < IMAG_LIMIT` и проверку поля; цветные линии — подтверждённые ветви.
На Im(n_eff) отмечены ±IMAG_LIMIT, вертикально — опорные длины волн. Исходные точки сохранены без фильтрации.

Порог применяется строго **на каждой длине волны**, включая ручной выбор опорных решений.
`mode` — внутренний номер от 0 по возрастанию Re(n_eff) в опоре; `solnum` — номер решения COMSOL.
Это не обозначения TE/TM. Для IS требуются ровно две ветви: 0 = signal, 1 = idler.
Если проверка не пройдена, причина будет в `pairs`, а точки останутся на графике.

In [ ]:
from linbo3.storage import analysis_root
from IPython.display import Image, display
from linbo3.saved_modes import inspect_modes

IMAG_LIMIT = 1e-6
MIN_OVERLAP = 0.9
MIN_MARGIN = 0.1
ANCHOR_SOLNUMS = None  # None: из манифеста; {}: полностью автоматически
# Пример явного выбора: {"0.7": {"PUMP": [3, 1], "IS": [2, 4]}}

tracking = dict(imag_limit=IMAG_LIMIT, min_overlap=MIN_OVERLAP, min_margin=MIN_MARGIN)
inspection = None
if result_dir is None:
    print("Сначала загрузи готовые данные в разделе 2.")
else:
    inspection = inspect_modes(result_dir, output_dir=analysis_root(result_dir) / "mode_diagnostics_poly6",
                               tracking=tracking, anchor_solnums=ANCHOR_SOLNUMS)
    for width in sorted(inspection.pairs["width_um"].unique()):
        display(Image(filename=str(inspection.output_dir / f"w_{width:g}" / "neff_raw.png")))

## 4. Выбор степени полинома и производные по ω

Задай POLYNOMIAL_DEGREE (целое число 1–6) в следующей ячейке. По умолчанию выбрана степень 6. Аппроксимируется Re(n_eff) по ω; эта же выполненная аппроксимация определяет степень для JSI и K. После изменения степени повтори разделы 4–6. Сравнение степеней остаётся в отдельном ноутбуке.

In [ ]:
from linbo3.polynomial import fit_study
from linbo3.study_plots import StudyPlots

# Явный выбор степени единственного полинома по ω. Рабочее значение — 6.
POLYNOMIAL_DEGREE = 6
dispersion_study = None

if inspection is None:
    print("Сначала выполни контроль мод в разделе 3.")
elif not inspection.modes:
    print("Нет подтверждённых ветвей. Проверь pairs.csv в папке mode_diagnostics_poly6.")
else:
    complete_modes = {width: bands for width, bands in inspection.modes.items()
                      if all(band in bands for band in ("PUMP", "IS"))}
    if complete_modes:
        manifest = json.loads((result_dir / "config.json").read_text(encoding="utf-8"))
        dispersion_study = fit_study(complete_modes, degrees=(POLYNOMIAL_DEGREE,), pump_wavelength_um=manifest["pump_wavelength_um"])
        plots = StudyPlots(dispersion_study, inspection.output_dir / "figures")
        plots.plot_dispersion_by_width()

## 5. JSI и K выбранной степени, 2000 фс

Степень берётся из выполненной аппроксимации раздела 4. Одна общая фигура: строки — ширины по возрастанию, столбцы — PUMP-ранги слева направо. Ширины без подтверждённых PUMP/IS не включаются в фигуру. Для каждого ряда — отдельный colorbar. Отсутствующая PUMP-мода в допустимом ряду отмечается пустой панелью. RUN_STUDY=True рассчитывает данные с переиспользованием кэша; False открывает сохранённую фигуру.

In [ ]:
import importlib
import linbo3.single_degree as single_degree

importlib.reload(single_degree)

run_single_study = single_degree.run_single_study
single_directory = single_degree.single_directory
saved_single_is_current = single_degree.saved_single_is_current

# Степень берётся из фактически выполненной аппроксимации.
RUN_STUDY = False
STUDY_POINTS = 1501
study_dir = None
comparison_ready = False

if result_dir is None or inspection is None or dispersion_study is None:
    print("Сначала выполни разделы 2–4.")
else:
    CALCULATION_DEGREE = dispersion_study.degrees[0]
    study_dir = single_directory(result_dir, STUDY_POINTS, degree=CALCULATION_DEGREE)
    if RUN_STUDY:
        study_dir = run_single_study(result_dir, points=STUDY_POINTS, tracking=tracking,
                                     anchor_solnums=ANCHOR_SOLNUMS, degree=CALCULATION_DEGREE)
    # Перерисовка из готового кэша: без повторного расчёта спектров и K.
    numerical_ready = saved_single_is_current(
        result_dir, inspection, STUDY_POINTS, degree=CALCULATION_DEGREE, require_overview=False,
    )
    overview_ready = saved_single_is_current(result_dir, inspection, STUDY_POINTS, degree=CALCULATION_DEGREE)
    if numerical_ready and not overview_ready:
        try:
            single_degree.refresh_single_overview(result_dir, inspection, STUDY_POINTS, degree=CALCULATION_DEGREE)
        except FileNotFoundError:
            print("Не все файлы кэша JSI найдены. Установи RUN_STUDY = True.")
    comparison_ready = saved_single_is_current(result_dir, inspection, STUDY_POINTS, degree=CALCULATION_DEGREE)
    if not comparison_ready:
        print(f"Нет актуальной общей фигуры JSI и K для степени {CALCULATION_DEGREE}. Установи RUN_STUDY = True.")
    else:
        display(Image(filename=str(study_dir / "figures" / "jsi_overview.png")))

## 6. Число Шмидта от ширины — выбранная степень

Каждая панель показывает PUMP-ранг. Ранг 1 соответствует наибольшему Re(n_eff) в опорной точке отдельно на каждой ширине. Поля между ширинами не сопоставляются; пропущенные случаи остаются разрывами.

In [ ]:
import importlib
import json
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import linbo3.single_degree as single_degree

# Числа K читаются из сохранённого расчёта, независимо от наличия общей картинки JSI.
# Перечитываем модуль, чтобы ячейка работала и после обновления кода в открытом ядре.
importlib.reload(single_degree)

if globals().get("result_dir") is None or globals().get("inspection") is None or globals().get("dispersion_study") is None:
    print("Сначала выполни разделы 2–4: нужны данные, контроль мод и выбранная степень аппроксимации.")
else:
    degree = dispersion_study.degrees[0]
    points = globals().get("STUDY_POINTS", 1501)
    k_dir = single_degree.single_directory(result_dir, points, degree=degree)
    summary_file = k_dir / "summary.csv"
    numerical_ready = single_degree.saved_single_is_current(
        result_dir, inspection, points, degree=degree, require_overview=False,
    )
    if not numerical_ready or not summary_file.is_file():
        print(f"Нет актуальных численных результатов K для степени {degree} и сетки {points}. "
              "В разделе 5 установи RUN_STUDY = True и выполни расчёт.")
    else:
        values = pd.read_csv(summary_file)
        if values.empty:
            print("В summary.csv нет рассчитанных случаев. Проверь pairs.csv в папке расчёта.")
        elif not values["degree"].eq(degree).all() or not values["N_POINTS"].eq(points).all():
            raise ValueError("Степень или сетка в summary.csv не соответствует текущему выбору. Повтори раздел 5.")
        else:
            # pump_label уже содержит ранг по убыванию Re(n_eff) в опоре.
            # Не пересчитываем номера и не зависим от branches.csv или comparison_ready.
            values["pump_rank"] = values["pump_label"].str.extract(r"(\d+)$", expand=False).astype(int)
            if values.duplicated(["width_um", "pump_rank"]).any():
                raise ValueError("В summary.csv повторяются пары ширина / PUMP-ранг.")
            manifest = json.loads((result_dir / "config.json").read_text(encoding="utf-8-sig"))
            widths = sorted(manifest["widths_um"])
            ranks = sorted(values["pump_rank"].unique())
            fig, axes = plt.subplots((len(ranks) + 1) // 2, 2,
                                     figsize=(12, 3.7 * ((len(ranks) + 1) // 2)),
                                     sharex=True, squeeze=False, layout="constrained")
            for ax, rank in zip(axes.flat, ranks):
                part = values.loc[values["pump_rank"].eq(rank)].set_index("width_um").reindex(widths)
                ax.plot(widths, part["K"], "o-", lw=1.4, ms=4, label=f"Степень {degree}")
                ax.set(title=f"PUMP-мода {rank:g}", xlabel="Ширина, мкм", ylabel="Число Шмидта K", xticks=widths)
                ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
                ax.grid(alpha=.25)
                ax.legend()
            for ax in axes.flat[len(ranks):]:
                ax.set_visible(False)
            fig.suptitle(f"Число Шмидта от ширины · степень {degree} · 2000 фс")
            fig.savefig(k_dir / "schmidt_number_vs_width.png", dpi=160)
            plt.show()
            plt.close(fig)
